W&B = experiment tracking dashboard

It helps you:

✅ Log metrics

✅ Track hyperparameters

✅ Compare experiments

✅ Save models

✅ Share results with links

✅ Make experiments reproducible

Core idea:

W&B remembers every experiment so you don't rely on memory or scattered notebook outputs.

In [4]:
# import wandb

# # 1. Start a run — call once at the top
# wandb.init(
#     project="cifar10-classifier",
#     name="resnet18-finetune",
#     config={
#         "model":          "resnet18",
#         "epochs":         15,
#         "batch_size":     64,
#         "lr_head":        1e-3,
#         "lr_backbone":    1e-4,
#         "optimizer":      "adam",
#         "augmentation":   True,
#     }
# )

# # 2. Log metrics each epoch — call inside training loop
# wandb.log({
#     "epoch":        epoch,
#     "train/loss":   train_loss,
#     "train/acc":    train_acc,
#     "val/loss":     val_loss,
#     "val/acc":      val_acc,
#     "lr":           optimizer.param_groups[0]['lr'],
# })

# # 3. Log images — sample predictions
# wandb.log({"predictions": wandb.Image(fig)})

# # 4. Finish the run — call once at the end
# wandb.finish()

In [5]:
# import torch, torchvision, wandb
# import torchvision.models as models
# import torchvision.transforms as transforms
# import torch.nn as nn, torch.optim as optim
# import matplotlib.pyplot as plt
# import numpy as np

# # --- Config ---
# config = dict(
#     epochs=15, batch_size=64,
#     lr_head=1e-3, lr_backbone=1e-4,
#     weight_decay=1e-4, resize=64,
# )

# wandb.init(project="cifar10-classifier",
#            name="resnet18-final", config=config)
# cfg = wandb.config

# device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# classes = ['plane','car','bird','cat','deer',
#            'dog','frog','horse','ship','truck']

# # --- Data ---
# tfm_tr = transforms.Compose([
#     transforms.Resize(cfg.resize),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomCrop(cfg.resize, padding=8),
#     transforms.ToTensor(),
#     transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
# ])
# tfm_val = transforms.Compose([
#     transforms.Resize(cfg.resize), transforms.ToTensor(),
#     transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
# ])
# tr_data  = torchvision.datasets.CIFAR10('./data',True, transform=tfm_tr)
# val_data = torchvision.datasets.CIFAR10('./data',False,transform=tfm_val)
# tr_ldr   = torch.utils.data.DataLoader(tr_data, cfg.batch_size, shuffle=True,  num_workers=0)
# val_ldr  = torch.utils.data.DataLoader(val_data,cfg.batch_size, shuffle=False, num_workers=0)

# # --- Model ---
# model    = models.resnet18(weights='IMAGENET1K_V1')
# model.fc = nn.Sequential(nn.Linear(512,256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(256,10))
# model    = model.to(device)
# crit     = nn.CrossEntropyLoss()
# opt      = optim.Adam([
#     {'params': model.fc.parameters(),     'lr': cfg.lr_head},
#     {'params': [p for n,p in model.named_parameters() if 'fc' not in n],
#                'lr': cfg.lr_backbone},
# ], weight_decay=cfg.weight_decay)
# sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)

# # --- Log model architecture ---
# wandb.watch(model, log='all', log_freq=100)

# best_val_acc = 0
# for epoch in range(cfg.epochs):
#     # Train
#     model.train()
#     tr_loss, tr_correct, tr_total = 0, 0, 0
#     for imgs, lbls in tr_ldr:
#         imgs, lbls = imgs.to(device), lbls.to(device)
#         opt.zero_grad()
#         out  = model(imgs)
#         loss = crit(out, lbls)
#         loss.backward(); opt.step()
#         tr_loss += loss.item()
#         _, preds = torch.max(out,1)
#         tr_correct += (preds==lbls).sum().item()
#         tr_total   += lbls.size(0)
#     sched.step()

#     # Validate
#     model.eval()
#     vl_loss, vl_correct, vl_total = 0, 0, 0
#     with torch.no_grad():
#         for imgs, lbls in val_ldr:
#             imgs, lbls = imgs.to(device), lbls.to(device)
#             out  = model(imgs)
#             loss = crit(out, lbls)
#             vl_loss += loss.item()
#             _, preds = torch.max(out,1)
#             vl_correct += (preds==lbls).sum().item()
#             vl_total   += lbls.size(0)

#     ta = tr_correct/tr_total
#     va = vl_correct/vl_total

#     # Log to W&B
#     wandb.log({
#         "epoch": epoch+1,
#         "train/loss": tr_loss/len(tr_ldr),
#         "train/acc":  ta,
#         "val/loss":   vl_loss/len(val_ldr),
#         "val/acc":    va,
#         "lr":         opt.param_groups[0]['lr'],
#     })

#     print(f"Epoch {epoch+1:2d} | train {ta:.3f} | val {va:.3f}")

#     if va > best_val_acc:
#         best_val_acc = va
#         torch.save(model.state_dict(), 'best_resnet18.pth')
#         wandb.save('best_resnet18.pth')   # sync model file to W&B

# print(f"\nBest: {best_val_acc:.3f}")

Quick Summary

Only 4 W&B ideas matter:

| Call             | Purpose           |
| ---------------- | ----------------- |
| `wandb.init()`   | Start experiment  |
| `wandb.log()`    | Save metrics      |
| `wandb.Image()`  | Save plots/images |
| `wandb.finish()` | Close run         |

Extra:

wandb.watch() → monitor model
wandb.save() → upload files

Core idea:

W&B turns training from scattered print statements into a tracked, reproducible experiment dashboard.

In [6]:
# from sklearn.metrics import confusion_matrix
# import seaborn as sns

# model.eval()
# all_preds, all_labels = [], []
# with torch.no_grad():
#     for imgs, lbls in val_ldr:
#         imgs = imgs.to(device)
#         _, preds = torch.max(model(imgs), 1)
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(lbls.numpy())

# cm = confusion_matrix(all_labels, all_preds)
# fig, ax = plt.subplots(figsize=(10,8))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=classes, yticklabels=classes, ax=ax)
# ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
# ax.set_title('Confusion Matrix — ResNet18 CIFAR-10')
# plt.tight_layout()

# # Log to W&B
# wandb.log({"confusion_matrix": wandb.Image(fig)})
# plt.show()

In [7]:
# dataiter = iter(val_ldr)
# imgs, lbls = next(dataiter)
# imgs_dev = imgs.to(device)

# model.eval()
# with torch.no_grad():
#     outputs = model(imgs_dev)
#     probs   = torch.softmax(outputs, dim=1)
#     _, preds = torch.max(outputs, 1)

# # Log first 16 predictions as W&B table
# table = wandb.Table(columns=["image","true","predicted","confidence"])
# for i in range(16):
#     img_unnorm = imgs[i] * 0.5 + 0.5   # unnormalise
#     table.add_data(
#         wandb.Image(img_unnorm.permute(1,2,0).numpy()),
#         classes[lbls[i]],
#         classes[preds[i].item()],
#         f"{probs[i][preds[i]].item():.1%}"
#     )

# wandb.log({"predictions_table": table})
# wandb.finish()

# print("Run complete. View at: https://wandb.ai")

Big Picture

After this step W&B becomes a complete experiment dashboard.

You now have:

Metrics

Train loss

Val loss

Accuracy

Diagnostics

Confusion matrix

Class-wise mistakes

Visual evidence

Real predictions

Confidence scores

Example images

This is why W&B matters.

Without W&B:

"My model got 81%."

With W&B:

"Here's the training history, mistakes, confusion matrix, and prediction examples."

That difference is what makes a project feel closer to professional ML work.